In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "modules" / "minio"))

from minio_spark import MinioSparkClient
from os import getenv
from pyspark.sql import functions as F

In [2]:
client = MinioSparkClient(
    "https://minio.fdi.ucm.es",
    getenv("MINIO_ACCESS_KEY"),
    getenv("MINIO_SECRET_KEY"),
    "pd2/cityenjoyer",
    memory = 8,
    heapsize = 4,
    num_part = 500
)
client.connect()

:: loading settings :: url = jar:file:/home/QuiSioU/UCM/y3/PD2/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/QuiSioU/.ivy2.5.2/cache
The jars for the packages stored in: /home/QuiSioU/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-19b69a56-5613-4ca6-b303-e3bfdf3ff3bc;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.1 in central
	found software.amazon.awssdk#bundle;2.24.6 in central
	found org.wildfly.openssl#wildfly-openssl;1.1.3.Final in central
:: resolution report :: resolve 109ms :: artifacts dl 4ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.4.1 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.1.3.Final from central in [default]
	software.amazon.awssdk#bundle;2.24.6 from central in [default]
	---------------------------------------------------------------------
	|     

In [3]:
df = client.read_parquet("21-25_merged.parquet")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [ ]:
print("Calculating thresholds from sample...")
df_sample = df.sample(False, 0.02, seed=42) 

stats_exprs = []
for col in ["trip_distance", "fare_amount", "tip_amount", "tolls_amount", "total_amount"]:
    stats_exprs.append(F.expr(f"percentile_approx({col}, array(0.25, 0.5, 0.75))").alias(f"{col}_stats"))

df_stats = df_sample.groupBy("PULocationID").agg(*stats_exprs).collect()

stats_rows = []
for row in df_stats:
    pid = row["PULocationID"]
    row_data = {"PULocationID": pid}
    for col in ["trip_distance", "fare_amount", "tip_amount", "tolls_amount", "total_amount"]:
        q1, med, q3 = row[f"{col}_stats"]
        iqr = q3 - q1
        row_data[f"{col}_low"] = q1 - 3.0 * iqr
        row_data[f"{col}_high"] = q3 + 3.0 * iqr
        row_data[f"{col}_med"] = med
    stats_rows.append(row_data)

broadcast_stats = client._spark.createDataFrame(stats_rows)

print("Cleaning full dataset...")
df_joined = df.join(F.broadcast(broadcast_stats), on="PULocationID", how="left")

df_final = df_joined.select(
    *[F.col(c) for c in df.columns if c not in ("trip_distance", "fare_amount", "tip_amount", "tolls_amount", "total_amount")],
    *[
        F.when(
            (F.col(col) < F.col(f"{col}_low")) | (F.col(col) > F.col(f"{col}_high")),
            F.col(f"{col}_med")
        ).otherwise(F.col(col)).alias(col)
        for col in ("trip_distance", "fare_amount", "tip_amount", "tolls_amount", "total_amount")
    ]
)

print("Savingto MinIO...")

df_final.write.mode("overwrite").format("parquet").save("../data/21-25_cleaned.parquet")

Calculating thresholds from sample...


Cleaning full dataset...
Savingto MinIO...
